In [ ]:
### PACKAGES ###

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.transforms import Compose, Resize, Grayscale, ToTensor, Normalize, RandomHorizontalFlip, RandomVerticalFlip, RandomRotation, RandomAffine
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader


import numpy as np
import time
import pandas as pd
import matplotlib.pyplot as plt
import os
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE

from tqdm import tqdm
from glob import glob

from pathlib import Path

In [ ]:
### INPUTS ###

# Define the output directory containing segmented images
input_folder = ""

# where to save the results
output_folder = ""
os.makedirs(output_folder, exist_ok=True)

In [ ]:
### CONVOLUTIONAL AUTOENCODER Architecture #### 

latent_dim = 32
input_channel = 1

encoder = nn.Sequential(
    nn.Conv2d(input_channel, 32, 3, padding=1), 
    nn.ReLU(),
    nn.MaxPool2d(2),  # 32x32

    nn.Conv2d(32, 64, 3, padding=1), 
    nn.ReLU(),
    nn.MaxPool2d(2),  # 16x16

    nn.Conv2d(64, 128, 3, padding=1), 
    nn.ReLU(),
    nn.MaxPool2d(2),  # 8x8

    nn.Conv2d(128, 256, 3, padding=1), 
    nn.ReLU(),
    nn.MaxPool2d(2),  # 4x4

    # dense top layers
    nn.Flatten(),                       # 4096
    nn.Linear(4096, 512), nn.ReLU(),
    nn.Linear(512, latent_dim)          # 32D features
)

decoder = nn.Sequential(
    nn.Linear(latent_dim, 512), nn.ReLU(),
    nn.Linear(512, 4096), nn.ReLU(),
    nn.Unflatten(1, (256, 4, 4)),
    nn.ConvTranspose2d(256, 128, 2, stride=2), nn.ReLU(),  # 8x8
    nn.ConvTranspose2d(128, 64, 2, stride=2), nn.ReLU(),   # 16x16
    nn.ConvTranspose2d(64, 32, 2, stride=2), nn.ReLU(),    # 32x32
    nn.ConvTranspose2d(32, 1, 2, stride=2),                # 64x64
    nn.Tanh()  # or nn.Sigmoid() if inputs in [0,1]
)

# Optimizer and loss function
optimizer = torch.optim.Adam(
    list(encoder.parameters()) + list(decoder.parameters()),
    lr=1e-3, 
    betas=(0.9, 0.999), 
    weight_decay=1e-4
)

loss_fn = nn.SmoothL1Loss(beta=0.1) # Huber loss for robustness


In [ ]:
# Load models from previous training

encoder_path = '/Volumes/PortableSSD/Bence_B/monitoring/results_test_2/ConvAE/epoch_1_50/last_encoder.pth'
encoder.load_state_dict(torch.load(encoder_path))
print(f"Encoder loaded from files: {encoder_path}")

decoder_path = '/Volumes/PortableSSD/Bence_B/monitoring/results_test_2/ConvAE/epoch_1_50/last_decoder.pth'
decoder.load_state_dict(torch.load(decoder_path))
print(f"Decoder loaded from files: {decoder_path}")

# Optimizer and loss function
optimizer = torch.optim.Adam(
    list(encoder.parameters()) + list(decoder.parameters()),
    lr=1e-3, 
    betas=(0.9, 0.999), 
    weight_decay=1e-4
)

loss_fn = nn.SmoothL1Loss(beta=0.1) # Huber loss for robustness

In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class ImagePathDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        """
        Dataset from list of image file paths.
        
        Args:
            image_paths: List of image file paths
            transform: Optional transform to apply to images
        """
        self.image_paths = image_paths
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')  # or 'L' for grayscale
        
        if self.transform:
            image = self.transform(image)
        
        return image, img_path  # Returns image tensor and path

def create_dataloader_from_paths(image_paths, transform=None, batch_size=64, shuffle=False, num_workers=0):
    """
    Create a DataLoader from a list of image paths.
    
    Args:
        image_paths: List of image file paths
        transform: Torchvision transforms to apply
        batch_size: Batch size for DataLoader
        shuffle: Whether to shuffle the data
        num_workers: Number of worker processes for loading
    
    Returns:
        DataLoader object
    """
    dataset = ImagePathDataset(image_paths, transform=transform)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=num_workers)
    
    return dataloader

In [ ]:
### Randomly select 10 percent of the data and create train and validation set ### 
### find segments in output files 

# found f_batch_XXX folders
fbatch_folder_paths = [
    path for path in glob.glob(os.path.join(input_folder, 'ROIs', '**', 'f_batch_**', 'f_batch_*'), recursive=True)
    if os.path.isdir(path)
]

# iterate through each f_batch_XXX folder and collect all png images
images = []
for fbatch_folder in fbatch_folder_paths:
    images += glob.glob(os.path.join(fbatch_folder, '*.png'))

print(f"Total images found: {len(images)}")

# pick random 10 percent of images and create tensor data loader
import random

sampled_images = random.sample(images, k=int(len(images) * 0.1))
print(f"Sampled images: {len(sampled_images)}")

train_images = sampled_images[:int(0.8 * len(sampled_images))]
val_images = sampled_images[int(0.8 * len(sampled_images)):]

In [ ]:
### load images from saved paths

log_path = ""

train_images = []
with open(os.path.join(log_path, 'train_image_paths.txt'), 'r') as f:
    for line in f:
        train_images.append(line.strip())
        
val_images = []
with open(os.path.join(log_path, 'val_image_paths.txt'), 'r') as f:
    for line in f:
        val_images.append(line.strip())

print(f"Train images: {len(train_images)}, Val images: {len(val_images)}")



In [ ]:
### Image preparations ### 

# Training transform with augmentation
train_transform = Compose([
    Resize((64, 64)),
    Grayscale(num_output_channels=1),
    RandomHorizontalFlip(p=0.5),           # flip jellyfish horizontally
    RandomVerticalFlip(p=0.5),             # flip jellyfish vertically
    RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1)),  # shift & zoom
    ToTensor(),
    Normalize(mean=[0.5], std=[0.5])
])

# Val/Test transform (NO augmentation)
eval_transform = Compose([
    Resize((64, 64)),
    Grayscale(num_output_channels=1),
    ToTensor(),
    Normalize(mean=[0.5], std=[0.5])
])

# Create data loaders from image paths
train_loader = DataLoader(ImagePathDataset(train_images, transform=train_transform), batch_size=64, shuffle=True)
val_loader = DataLoader(ImagePathDataset(val_images, transform=eval_transform), batch_size=64, shuffle=False)


In [ ]:
### Check on data format ###
# Visualize some images with labels from the validation set

def image_examples(dataset ):

    fig, axs = plt.subplots(3, 6, figsize=(16, 8))
    for ax in axs.ravel():
        # Pick random image
        idx = np.random.randint(0, len(dataset))
        image,_ = dataset[idx]

        # Convert from tensor (C,H,W) -> (H,W,C)
        image_np = image.permute(1, 2, 0).numpy()

        # Show image
        ax.imshow(image_np, cmap = 'gray')
        
    plt.tight_layout()
    plt.show()

image_examples(train_loader.dataset)

In [ ]:
### TRAINING LOOP ###


# Device setup
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
#device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

encoder.to(device)
decoder.to(device)

epochs = 50
train_losses = []
val_losses = []
time_per_epoch = []
best_val_loss = float("inf")

for epoch in range(epochs):
    start_time = time.time()
    
    print(f"\nEpoch {epoch+1}/{epochs}")
    print("-" * 40)
    
    ### TRAINING PHASE ###
    encoder.train()
    decoder.train()
    
    running_train_loss = 0.0
    num_batches = len(train_loader)
    
    for batch_idx, (images, _) in enumerate(train_loader):  # Ignore labels
        images = images.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass: encode then decode
        z = encoder(images)
        recon = decoder(z)
        
        # Compute reconstruction loss
        loss = loss_fn(recon, images)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        running_train_loss += loss.item()
        
        if batch_idx % 10 == 0:
            print(f"Batch {batch_idx}/{num_batches}, Loss: {loss.item():.4f}")
    
    avg_train_loss = running_train_loss / num_batches
    train_losses.append(avg_train_loss)
    print("-" * 40)
    print(f"Training Loss: {avg_train_loss:.4f}")
    
    ### VALIDATION PHASE ###
    encoder.eval()
    decoder.eval()
    
    running_val_loss = 0.0
    num_val_batches = len(val_loader)
    
    with torch.no_grad():
        for images, _ in val_loader:
            images = images.to(device)
            
            z = encoder(images)
            recon = decoder(z)
            
            loss = loss_fn(recon, images)
            running_val_loss += loss.item()
    
    avg_val_loss = running_val_loss / num_val_batches
    val_losses.append(avg_val_loss)
    print(f"Validation Loss: {avg_val_loss:.4f}")
    
    # Save best model
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(encoder.state_dict(), os.path.join(output_folder, "best_encoder.pth"))
        torch.save(decoder.state_dict(), os.path.join(output_folder, "best_decoder.pth"))
        print(f"Best model saved (val_loss: {best_val_loss:.4f})")
    
    elapsed = time.time() - start_time
    time_per_epoch.append(elapsed)
    print(f"Epoch time: {elapsed:.1f}s")

# Save final models
torch.save(encoder.state_dict(), os.path.join(output_folder, "last_encoder.pth"))
torch.save(decoder.state_dict(), os.path.join(output_folder, "last_decoder.pth"))

# Save training log
log_df = pd.DataFrame({
    "epoch": list(range(1, epochs + 1)),
    "train_loss": train_losses,
    "val_loss": val_losses,
    "time_sec": time_per_epoch
})
log_df.to_csv(os.path.join(output_folder, "training_log.csv"), index=False)

# Plot losses
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Autoencoder Training")
plt.grid(True)
plt.savefig(os.path.join(output_folder, "loss_plot.png"))
plt.show()

print("\nTraining complete!")


In [ ]:
### Use decoder to reconstruct some test images and visualize ###

encoder_path = ""
encoder.load_state_dict(torch.load(encoder_path))
print(f"Encoder loaded from files: {encoder_path}")

decoder_path = ""
decoder.load_state_dict(torch.load(decoder_path))
print(f"Decoder loaded from files: {decoder_path}") 

#device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
encoder.to(device)
decoder.to(device)
encoder.eval()
decoder.eval()  

# Select random samples from val set
num_samples = 5
random_indices = np.random.choice(len(val_loader.dataset), size=num_samples, replace=False)
sample_images = []
for idx in random_indices:
    img, _ = val_loader.dataset[idx]
    sample_images.append(img.unsqueeze(0))  # add batch dim

sample_batch = torch.cat(sample_images, dim=0).to(device)  # (num_samples, 1, 64, 64)
with torch.no_grad():
    z = encoder(sample_batch)
    reconstructions = decoder(z)  # (num_samples, 1, 64, 64)

# Plot original vs reconstructed: top row original, bottom row reconstructed
fig, axes = plt.subplots(2, num_samples, figsize=(num_samples * 2, 4))
for i in range(num_samples):
    # Original
    axes[0, i].imshow(sample_batch[i].cpu().squeeze(), cmap='gray', vmin=-1, vmax=1)
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_title("Original", fontsize=12)
    
    # Reconstructed
    axes[1, i].imshow(reconstructions[i].cpu().squeeze(), cmap='gray', vmin=-1, vmax=1)
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_title("Reconstructed", fontsize=12)

In [ ]:
### EVALUATION ON VAL SET ###

# Load best models from files
encoder = encoder 
print(f"Encoder loaded from files: {encoder_path}")

device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
#device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

encoder.to(device)
encoder.eval()

# Extract features and collect labels
all_features = []
all_labels = []

with torch.no_grad():
    for images, _ in val_loader:
        images = images.to(device)
        z = encoder(images)  # 32D features
        all_features.append(z.cpu().numpy())

features = np.vstack(all_features)  # (N, 32)

print(f"Extracted {features.shape[0]} feature vectors of {features.shape[1]} dimensions")

In [ ]:
### K-mean clustering  ###

# select nmber of clusters based on silhouette score
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import torch.nn.functional as F

features = features = np.vstack(all_features)  # use unscaled features
#features = StandardScaler().fit_transform(features)  # scale features
#features = PCA(n_components=10).fit_transform(features)  # reduce to 10D for faster clustering
#features = F.normalize(torch.tensor(features), p=2, dim=1).numpy()

# Range of K to test
K_range = range(2, 50)
silhouette_scores = []

# Store inertia for each K
inertia_list = []

for k in K_range:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    
    # silhouette score
    labels = kmeans.fit_predict(features)
    score = silhouette_score(features, labels)
    silhouette_scores.append(score)
    # elbow mehthod
    kmeans.fit(features)
    inertia_list.append(kmeans.inertia_)

# Plot silhouette scores
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(K_range, silhouette_scores, marker='o')
plt.xlabel('Number of clusters (K)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Analysis on Features')
plt.grid(True)

### ELBOW METHOD ###
    
# Plot the elbow curve
plt.subplot(1, 2, 2)
plt.plot(K_range, inertia_list, marker='o')
plt.xlabel('Number of clusters (K)')
plt.ylabel('Inertia (Within-cluster Sum of Squares)')
plt.title('Elbow Method to Determine Optimal K')
plt.grid(True)
plt.show()

best_k = K_range[silhouette_scores.index(max(silhouette_scores))]
print(f"Best K according to silhouette score: {best_k}")


In [ ]:
### Visualisation of clustering ###

from sklearn.decomposition import PCA
pca = PCA(n_components=2)

kmeans = KMeans(
    n_clusters=45, # select based on previous silhouette and elbow analysis
    random_state=42,
    n_init=10
)

cluster_labels = kmeans.fit_predict(features)

features_pca = pca.fit_transform(features)

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
scatter = plt.scatter(features_pca[:, 0], features_pca[:, 1],
                        c=cluster_labels, s=30, alpha=0.6,
                        edgecolors='black', linewidth=0.5)
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("PCA Visualization with Clusters")
plt.grid(True, alpha=0.3)
plt.tight_layout()

tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
features_2d = tsne.fit_transform(features)

plt.subplot(1, 2, 2)
scatter1 = plt.scatter(features_2d[:, 0], features_2d[:, 1], 
                          c=cluster_labels, s=30, alpha=0.6, 
                          edgecolors='black', linewidth=0.5)
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.title("t-SNE Visualization with Clusters")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.colorbar(scatter1, label="Cluster ID")
plt.show()

In [ ]:

### cluster examples visualization ###

for cluster_id in sorted(set(cluster_labels)):
    cluster_indices = np.where(cluster_labels == cluster_id)[0]
    n_examples = min(10, len(cluster_indices))
    sampled_indices = np.random.choice(cluster_indices, size=n_examples, replace=False)
    
    fig, axs = plt.subplots(1, n_examples, figsize=(n_examples * 3, 3))
    for i, idx in enumerate(sampled_indices):
        img_path = val_loader.dataset.imgs[idx][0]
        image = Image.open(img_path).convert('RGB')
        image = eval_transform(image)  # Apply eval transform
        image_np = image.permute(1, 2, 0).numpy()
        
        axs[i].imshow(image_np, cmap='gray')
        axs[i].axis('off')
    
    plt.suptitle(f"Cluster {cluster_id} Examples ({len(cluster_indices)} images)", fontsize=16)
    plt.tight_layout()
    plt.subplots_adjust(top=0.85)